<a href="https://colab.research.google.com/github/will-mccormack/CS-M148-Proj/blob/main/COM_SCI_M148_NN_with_genre.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
import pandas as pd
from skimage import io, transform
import numpy as np
import matplotlib.pyplot as plt
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, utils
from sklearn.preprocessing import StandardScaler

import warnings
warnings.filterwarnings("ignore")

In [2]:
train_filepath = "https://raw.githubusercontent.com/will-mccormack/CS-M148-Proj/main/Data/train.csv"
validation_filepath = "https://raw.githubusercontent.com/will-mccormack/CS-M148-Proj/main/Data/validation.csv"

train_data = pd.read_csv(train_filepath)
validation_data = pd.read_csv(train_filepath)

In [3]:
class SpotifyDataset(Dataset):
  def __init__(self, csv_file):
    self.data = pd.read_csv(csv_file) # load data
    self.data['explicit'] = self.data['explicit'].astype(int) # change T/F to 1/0 encoding
    self.data = pd.get_dummies(self.data, columns=['track_genre'], dtype=float) # one hot encode track_genre
    # split x and y
    self.x = self.data.drop(columns=['popularity']).values
    self.y = self.data['popularity'].values
    scaler = StandardScaler()
    self.x = scaler.fit_transform(self.x) #scale data

  def __len__(self):
    return len(self.data)

  def __getitem__(self, idx):
    features = self.x[idx]
    label = self.y[idx]
    features_tensor = torch.tensor(features, dtype=torch.float32)
    label_tensor = torch.tensor(label, dtype=torch.float32)
    return features_tensor, label_tensor

In [4]:
spotify_dataset = SpotifyDataset(csv_file=train_filepath)

In [5]:
# spotify to tensor verification

train_loader = DataLoader(dataset=spotify_dataset, batch_size=32, shuffle=True)
data_iter = iter(train_loader)
features, labels = next(data_iter)

print(f"Features batch shape: {features.shape}")
print(f"Labels batch shape: {labels.shape}")

input_size = features.shape[1]
print(f"Input (features) size: {input_size}")

Features batch shape: torch.Size([32, 127])
Labels batch shape: torch.Size([32])
Input (features) size: 127


# NN

In [7]:
model = nn.Sequential(
    nn.Linear(input_size,32),
    nn.ReLU(),
    nn.Linear(32,16),
    nn.ReLU(),
    nn.Linear(16,1)
)

loss_type = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

num_epochs = 50

for epoch in range(num_epochs):
  epoch_loss = 0.0

  for features, labels in train_loader:
    # forward pass, predicted
    predictions = model(features)
    # loss
    loss = loss_type(predictions, labels.view(-1,1))
    #backprop, gradients and update weights
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    #update loss
    epoch_loss += loss.item()

  print(f"Epoch {epoch+1}/{num_epochs}, Loss: {epoch_loss/len(train_loader)}")

print("Training finished")

Epoch 1/50, Loss: 455.1808295304181
Epoch 2/50, Loss: 370.3195637936676
Epoch 3/50, Loss: 368.1479683553544
Epoch 4/50, Loss: 366.0512326111345
Epoch 5/50, Loss: 364.2101004362846
Epoch 6/50, Loss: 362.27932729395616
Epoch 7/50, Loss: 360.3760643005371
Epoch 8/50, Loss: 358.57694835722015
Epoch 9/50, Loss: 356.90001462287466
Epoch 10/50, Loss: 355.51311770244865
Epoch 11/50, Loss: 353.559675388277
Epoch 12/50, Loss: 352.02325597248256
Epoch 13/50, Loss: 350.3877636299015
Epoch 14/50, Loss: 349.33686031747874
Epoch 15/50, Loss: 347.81492230120466
Epoch 16/50, Loss: 346.53139583425906
Epoch 17/50, Loss: 345.65283387548055
Epoch 18/50, Loss: 344.54665428011714
Epoch 19/50, Loss: 343.58792328859187
Epoch 20/50, Loss: 342.34179608996925
Epoch 21/50, Loss: 341.4828851496668
Epoch 22/50, Loss: 340.4621651145517
Epoch 23/50, Loss: 339.634254435842
Epoch 24/50, Loss: 338.8136785008175
Epoch 25/50, Loss: 338.26428577300806
Epoch 26/50, Loss: 337.37936570602466
Epoch 27/50, Loss: 336.647303694653